# Eksperyment B - degradacja czestotliwosciowa synthetic

Notebook Colab do odpalenia eksperymentu B: trening YOLOv10n na synthetic 10k po degradacji obrazu i ewaluacja na realnym holdoucie.

Warianty:
- B1: blur + Gaussian noise,
- B2: Gaussian noise,
- B3: blur + Gaussian noise + JPEG.

Wyniki zapisuja sie do `results/per_size/expB*.json` oraz `results/baselines/expB*.json`.

In [1]:
# 0. GPU sanity check
# Jesli ta komorka nie przejdzie: Runtime -> Change runtime type -> GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "Brak GPU w runtime Colaba. Wlacz: Runtime -> Change runtime type -> GPU."
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

NVIDIA GeForce GTX 1060 6GB, 6144 MiB
CUDA: 12.4
GPU: NVIDIA GeForce GTX 1060 6GB


In [2]:
# 1. Repo + zaleznosci
# Jesli pracujesz na swoim forku/branchu, zmien REPO_URL i opcjonalnie REPO_BRANCH.
# Wazne: skrypty eksperymentu B musza byc juz commitniete i wypchniete na ten URL.
REPO_URL = "https://github.com/milosz-amg/rareplanes-domain-shift.git"
REPO_BRANCH = ""
REPO_DIR = "rareplanes-domain-shift"

import os
from pathlib import Path

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
if REPO_BRANCH:
    !git fetch origin {REPO_BRANCH}
    !git checkout {REPO_BRANCH}
!git pull --ff-only || true

required = [
    "src/make_frequency_degraded_dataset.py",
    "src/sweep_B_frequency.sh",
]
missing = [p for p in required if not Path(p).exists()]
assert not missing, "Brakuje skryptow eksperymentu B w sklonowanym repo: " + ", ".join(missing)

!pip -q install ultralytics pycocotools pillow matplotlib pandas tabulate

Cloning into 'rareplanes-domain-shift'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 132 (delta 56), reused 112 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 4.78 MiB | 19.49 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/home/milosz/rareplanes-domain-shift/notebooks/rareplanes-domain-shift


/home/milosz/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Already up to date.
ERROR: Operation cancelled by user
^C


## Dane dla eksperymentu B

Potrzebujemy tylko:
- adnotacji synthetic train/test do konwersji COCO -> YOLO,
- adnotacji real test i real test tiles do ewaluacji,
- synthetic 10k z list `configs/synthetic_10k_*_files.txt`.

Nie pobieramy real train, bo w eksperymencie B model nie widzi realnych danych treningowych.

In [3]:
# 2. Adnotacje + real test
from pathlib import Path

B = "https://rareplanes-public.s3.amazonaws.com"
for d in [
    "data/real/annotations",
    "data/real/tarballs",
    "data/synthetic/annotations",
    "data/synthetic/images/train",
    "data/real/PS-RGB_tiled",
]:
    Path(d).mkdir(parents=True, exist_ok=True)

!test -s data/real/annotations/instances_test_aircraft.json || curl -sf -o data/real/annotations/instances_test_aircraft.json {B}/real/metadata_annotations/instances_test_aircraft.json
!test -s data/synthetic/annotations/instances_train_aircraft.json || curl -sf -o data/synthetic/annotations/instances_train_aircraft.json {B}/synthetic/metadata_annotations/instances_train_aircraft.json
!test -s data/synthetic/annotations/instances_test_aircraft.json || curl -sf -o data/synthetic/annotations/instances_test_aircraft.json {B}/synthetic/metadata_annotations/instances_test_aircraft.json

!test -s data/real/tarballs/test.tar.gz || curl -fL -o data/real/tarballs/test.tar.gz {B}/real/tarballs/test/RarePlanes_test_PS-RGB_tiled.tar.gz
!tar -xzf data/real/tarballs/test.tar.gz -C data/real/PS-RGB_tiled/

real_test_tiles = len(list(Path("data/real/PS-RGB_tiled/PS-RGB_tiled").glob("*.png")))
print("real test tiles:", real_test_tiles)
assert real_test_tiles >= 2500, "Za malo real test tiles - sprawdz pobieranie tarballa."

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  889M  100  889M    0     0  8547k      0  0:01:46  0:01:46 --:--:-- 10.9M    0     0  8434k      0  0:01:47  0:00:57  0:00:50 8795k
real test tiles: 2710


In [4]:
# 3. Synthetic 10k przez HTTPS z resume
import os
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DST = Path("data/synthetic/images/train")
DST.mkdir(parents=True, exist_ok=True)
files = [l.strip() for l in open("configs/synthetic_10k_train_files.txt")]
files += [l.strip() for l in open("configs/synthetic_10k_val_files.txt")]
files = [f for f in files if f]
print("do pobrania:", len(files))

def fetch(fn):
    dst = DST / fn
    part = DST / (fn + ".part")
    if dst.exists() and dst.stat().st_size > 0:
        return True
    try:
        urllib.request.urlretrieve(f"{B}/synthetic/train/images/{fn}", part)
        os.replace(part, dst)
        return True
    except Exception:
        if part.exists():
            part.unlink()
        return False

ok = 0
with ThreadPoolExecutor(max_workers=32) as ex:
    for i, r in enumerate(ex.map(fetch, files), 1):
        ok += int(r)
        if i % 1000 == 0:
            print(f"  {i}/{len(files)} ok={ok}")

have = len(list(DST.glob("*.png")))
print(f"GOTOWE: ok={ok}/{len(files)}, na dysku={have}")
assert have >= int(len(files) * 0.99), "Za malo synthetic - uruchom komorke ponownie, resume dociagnie reszte."

do pobrania: 11764
  1000/11764 ok=1000
  2000/11764 ok=2000
  3000/11764 ok=2999
  4000/11764 ok=3999
  5000/11764 ok=4999
  6000/11764 ok=5998
  7000/11764 ok=6998
  8000/11764 ok=7998
  9000/11764 ok=8998
  10000/11764 ok=9998
  11000/11764 ok=10998
GOTOWE: ok=11762/11764, na dysku=11762


In [5]:
# 4. COCO -> YOLO + subsety synthetic
!python3 src/coco_to_yolo.py --domain synthetic --classes aircraft --val-frac 0.15
!python3 src/make_subset.py --n-train 10000 --name synthetic_10k
!python3 src/make_subset.py --n-train 1000 --name synthetic_1k

from pathlib import Path
syn_train = len(list(Path("data/yolo/synthetic_10k/images/train").glob("*.png")))
syn_val = len(list(Path("data/yolo/synthetic_10k/images/val").glob("*.png")))
syn1_train = len(list(Path("data/yolo/synthetic_1k/images/train").glob("*.png")))
syn1_val = len(list(Path("data/yolo/synthetic_1k/images/val").glob("*.png")))
print({"synthetic_10k_train": syn_train, "synthetic_10k_val": syn_val, "synthetic_1k_train": syn1_train, "synthetic_1k_val": syn1_val})
assert syn_train >= 9900, "synthetic_10k ma za malo obrazow train."

[synthetic/aircraft] train=9998 val=1764 test=0
  klasy (1): ['aircraft']
  -> /home/milosz/rareplanes-domain-shift/notebooks/rareplanes-domain-shift/data/yolo/synthetic_aircraft/data.yaml
[synthetic_10k] train=9998 val=1764 -> /home/milosz/rareplanes-domain-shift/notebooks/rareplanes-domain-shift/data/yolo/synthetic_10k/data.yaml
rozklad lotnisk (train):
   Atlanta_Airport         : 684
   Newark_Airport          : 677
   Miami_Airport           : 676
   SaltLakeCity_Airport    : 676
   Munich_Airport          : 676
   HongKong_Airport        : 673
   Zhukovsky_Airport       : 673
   Basel_Airport           : 672
   Oulu_Airport            : 672
   Unalaska_Airport        : 670
   Geneva_Airport          : 669
   Chicago_Airport         : 669
   SeoulIncheon_Airport    : 666
   London_Airport          : 665
   LosAngeles_Airport      : 580
[synthetic_1k] train=1000 val=176 -> /home/milosz/rareplanes-domain-shift/notebooks/rareplanes-domain-shift/data/yolo/synthetic_1k/data.yaml
rozkla

## Uruchomienie eksperymentu B

Jesli Colab wyrzuci OOM, zmniejsz `BATCH` z 64 na 32 albo 16 i odpal komorke jeszcze raz.

Na szybki smoke test ustaw `VARIANTS = "B1"`. Do finalnego uruchomienia ustaw `VARIANTS = "B1 B2 B3"`.

Skrypt zapisuje wyniki kazdego wariantu w `results/per_size/`.

In [7]:
# 5. Sweep B
# Smoke test: USE_SMOKE_DATASET = True, EPOCHS = 3, VARIANTS = "B1"
# Finalnie: USE_SMOKE_DATASET = False, EPOCHS = 60, VARIANTS = "B1 B2 B3"
USE_SMOKE_DATASET = False
EPOCHS = 60
BATCH = 32
WORKERS = 2
DEVICE = 0
VARIANTS = "B1 B2 B3"
CLEANUP_DATASETS = 

if USE_SMOKE_DATASET:
    SRC_DATASET = "data/yolo/synthetic_1k"
    DATASET_TAG = "1k_smoke"
else:
    SRC_DATASET = "data/yolo/synthetic_10k"
    DATASET_TAG = "10k"

!SRC_DATASET={SRC_DATASET} DATASET_TAG={DATASET_TAG} EPOCHS={EPOCHS} BATCH={BATCH} WORKERS={WORKERS} DEVICE={DEVICE} VARIANTS="{VARIANTS}" CLEANUP_DATASETS={CLEANUP_DATASETS} bash src/sweep_B_frequency.sh

[sweep B] SRC_DATASET=data/yolo/synthetic_10k DATASET_TAG=10k
[sweep B] EPOCHS=60 BATCH=32 WORKERS=2 DEVICE=0
[sweep B] VARIANTS=B1 B2 B3 CLEANUP_DATASETS=0
[B1] tworze dataset: blur + noise
[start] src=/home/milosz/rareplanes-domain-shift/notebooks/rareplanes-domain-shift/data/yolo/synthetic_10k
[start] dst=/home/milosz/rareplanes-domain-shift/notebooks/rareplanes-domain-shift/data/yolo/synthetic_10k_b1_blur_noise
[start] blur_radius<=0.4 noise_sigma<=5.0 jpeg_quality_min=None
[train] degradacja obrazow: 9998 -> /home/milosz/rareplanes-domain-shift/notebooks/rareplanes-domain-shift/data/yolo/synthetic_10k_b1_blur_noise/images/train
[train] 250/9998
[train] 500/9998
Traceback (most recent call last):
  File "/home/milosz/.local/lib/python3.10/site-packages/PIL/ImageFile.py", line 554, in _save
    fh = fp.fileno()
AttributeError: '_idat' object has no attribute 'fileno'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/hom

In [ ]:
# 6. Tabela wynikow B z JSON-ow
import json
from pathlib import Path
import pandas as pd

rows = []
for p in sorted(Path("results/per_size").glob("expB*_ml.json")):
    d = json.load(open(p))
    m = d.get("metrics", {})
    rows.append({
        "run": d.get("name", p.stem),
        "mAP@50": m.get("AP@.5"),
        "mAP@50:95": m.get("AP@[.5:.95]"),
        "AP_S": m.get("AP_small"),
        "AP_M": m.get("AP_medium"),
        "AP_L": m.get("AP_large"),
        "AR@100": m.get("AR@100"),
        "n_det": d.get("n_detections"),
        "file": str(p),
    })

df = pd.DataFrame(rows)
display(df)
Path("results").mkdir(exist_ok=True)
df.to_csv("results/expB_frequency_summary.csv", index=False)
df.to_markdown("results/expB_frequency_summary.md", index=False)
print("zapisano: results/expB_frequency_summary.csv/md")

In [ ]:
# 7. Pobranie wynikow B na komputer
from pathlib import Path
from google.colab import files
import zipfile

targets = []
targets += list(Path("results/per_size").glob("expB*_ml.json"))
targets += list(Path("results/baselines").glob("expB*_ml.json"))
for extra in ["results/expB_frequency_summary.csv", "results/expB_frequency_summary.md"]:
    p = Path(extra)
    if p.exists():
        targets.append(p)

zip_path = Path("expB_frequency_results.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for p in targets:
        z.write(p, p.as_posix())

print("pliki w paczce:")
for p in targets:
    print(" -", p)
files.download(str(zip_path))